 compute_physics_residuals 

| Feature | t1 | t2 | t3 | t4 | t5 |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **V (Voltage)** | -65.0 | -60.0 | -50.0 | -30.0 | 10.0 |
| **n (Gating)** | 0.31 | 0.32 | 0.35 | 0.40 | 0.50 |
| **m (Gating)** | 0.05 | 0.06 | 0.10 | 0.25 | 0.60 |
| **h (Gating)** | 0.60 | 0.58 | 0.50 | 0.35 | 0.15 |


In [37]:
using DataFrames

# Create a DataFrame with the columns mapped to vectors
u_hat = DataFrame(
    Feature = ["V (Voltage)", "n (Gating)", "m (Gating)", "h (Gating)"],
    t1 = [-65.0, 0.31, 0.05, 0.60],
    t2 = [-60.0, 0.32, 0.06, 0.58],
    # To get high theoretical derivatives, we must scale up our opening gates (m and n) 
    # and scale down our closing gate (h) at t3 and t4 to simulate an opening channel!
    t3 = [-50.0, 0.45, 0.75, 0.20], 
    t4 = [-30.0, 0.70, 0.95, 0.05],
    t5 = [10.0,  0.85, 0.99, 0.01]
)
dt = 0.01
# Display the DataFrame
println(u_hat)


4×6 DataFrame
 Row │ Feature      t1       t2       t3       t4       t5      
     │ String       Float64  Float64  Float64  Float64  Float64 
─────┼──────────────────────────────────────────────────────────
   1 │ V (Voltage)   -65.0    -60.0    -50.0    -30.0     10.0
   2 │ n (Gating)      0.31     0.32     0.45     0.7      0.85
   3 │ m (Gating)      0.05     0.06     0.75     0.95     0.99
   4 │ h (Gating)      0.6      0.58     0.2      0.05     0.01


In [38]:
V = @view u_hat[1, :]

Row,Feature,t1,t2,t3,t4,t5
,String,Float64,Float64,Float64,Float64,Float64
1,V (Voltage),-65.0,-60.0,-50.0,-30.0,10.0


In [45]:
V = @view u_hat[1, 2:6]
n = @view u_hat[2, 2:6]
m = @view u_hat[3, 2:6]
h = @view u_hat[4, 2:6]
println(V,n, m,h)

DataFrameRow
 Row │ t1       t2       t3       t4       t5      
     │ Float64  Float64  Float64  Float64  Float64 
─────┼─────────────────────────────────────────────
   1 │   -65.0    -60.0    -50.0    -30.0     10.0DataFrameRow
 Row │ t1       t2       t3       t4       t5      
     │ Float64  Float64  Float64  Float64  Float64 
─────┼─────────────────────────────────────────────
   2 │    0.31     0.32     0.45      0.7     0.85DataFrameRow
 Row │ t1       t2       t3       t4       t5      
     │ Float64  Float64  Float64  Float64  Float64 
─────┼─────────────────────────────────────────────
   3 │    0.05     0.06     0.75     0.95     0.99DataFrameRow
 Row │ t1       t2       t3       t4       t5      
     │ Float64  Float64  Float64  Float64  Float64 
─────┼─────────────────────────────────────────────
   4 │     0.6     0.58      0.2     0.05     0.01


In [47]:
V_numeric = Vector{Float32}(V[2:end])
n_numeric = Vector{Float32}(n[2:end])
m_numeric = Vector{Float32}(m[2:end])
h_numeric = Vector{Float32}(h[2:end])

4-element Vector{Float32}:
 0.58
 0.2
 0.05
 0.01

` Finite differernce `
$$\frac{dV}{dt} \approx \frac{V_{i+1} - V_{i-1}}{2\Delta t}$$

In [48]:
dV_dt_V_numeric  = (V_numeric[3:end] .- V_numeric[1:end-2])./(2f0* dt)
# dV_dt_n_numeric  = (n_numeric[3:end] .- n_numeric[1:end-2])./(2f0* dt)
# dV_dt_m_numeric  = (m_numeric[3:end] .- m_numeric[1:end-2])./(2f0* dt)
# dV_dt_h_numeric  = (h_numeric[3:end] .- h_numeric[1:end-2])./(2f0* dt)

println(dV_dt_V_numeric)

[1500.0, 3000.0]


`isolating the interior states ( t2 t3 t4 )`

In [49]:
V_int = @view V_numeric[2:end-1]
n_int = @view n_numeric[2:end-1]
m_int = @view m_numeric[2:end-1]
h_int = @view h_numeric[2:end-1]
println(V_int,n_int,m_int,h_int)

Float32[-50.0, -30.0]Float32[0.45, 0.7]Float32[0.75, 0.95]Float32[0.2, 0.05]


`  HH 4d model `

In [50]:
# --- Hodgkin-Huxley Physical Parameters ---
C_m   = 1.0f0      # Membrane Capacitance (uF/cm^2)
g_Na  = 120.0f0    # Max Conductance Sodium (mS/cm^2)
g_K   = 36.0f0     # Max Conductance Potassium (mS/cm^2)
g_L   = 0.3f0      # Max Conductance Leak (mS/cm^2)
E_Na  = 50.0f0     # Reversal Potential Sodium (mV)
E_K   = -77.0f0    # Reversal Potential Potassium (mV)
E_L   = -54.4f0    # Reversal Potential Leak (mV)
I_ext = 10.0f0     # External injected current
# -----------------------------------------------------------
# Voltage-Dependent Gating Functions (alpha and beta rates)
α_n(v) = 0.01f0 * (v + 55.0f0) / (1.0f0 - exp(-(v + 55.0f0) / 10.0f0) + 1f-6)
β_n(v) = 0.125f0 * exp(-(v + 65.0f0) / 80.0f0)

α_m(v) = 0.1f0 * (v + 40.0f0) / (1.0f0 - exp(-(v + 40.0f0) / 10.0f0) + 1f-6)
β_m(v) = 4.0f0 * exp(-(v + 65.0f0) / 18.0f0)

α_h(v) = 0.07f0 * exp(-(v + 65.0f0) / 20.0f0)
β_h(v) = 1.0f0 / (1.0f0 - exp(-(v + 35.0f0) / 10.0f0) + 1f-6)

β_h (generic function with 1 method)

`# 4. Compute Theoretical Derivatives F(u) from the physical equations`

In [51]:

# Voltage Equation: dV/dt = (I_ext - I_Na - I_K - I_L) / C_m
dV_dt_theo = (I_ext .- 
               g_Na .* (m_int.^3) .* h_int .* (V_int .- E_Na) .- 
               g_K  .* (n_int.^4) .* (V_int .- E_K) .- 
               g_L  .* (V_int .- E_L)) ./ C_m
               
# Gating Variable Equations: dx/dt = α_x(V)*(1 - x) - β_x(V)*x
dn_dt_theo = α_n.(V_int) .* (1.0f0 .- n_int) .- β_n.(V_int) .* n_int
dm_dt_theo = α_m.(V_int) .* (1.0f0 .- m_int) .- β_m.(V_int) .* m_int
dh_dt_theo = α_h.(V_int) .* (1.0f0 .- h_int) .- β_h.(V_int) .* h_int
println( " Theratical values ")
println( " Theratical values of V : ", dV_dt_theo)
println( " Theratical values of n :" , dn_dt_theo)
println( " Theratical values of m :", dm_dt_theo)
println( " Theratical values of h :", dm_dt_theo)

 Theratical values 
 Theratical values of V : Float32[981.3219, 7.9708014]
 Theratical values of n :Float32[0.023258023, 0.025212575]
 Theratical values of m :Float32[-1.1583004, -0.4645547]
 Theratical values of h :Float32[-1.1583004, -0.4645547]
